# OfficeAgentEnv - Layer 2 PPO Training (Colab)

This notebook trains a policy with PPO using **live environment interaction** (no static dataset, no heuristic policy).

## Sections
1. Setup
2. Model loading
3. Environment integration
4. Helpers (observation formatting + action parsing)
5. PPO setup
6. Baseline evaluation
7. PPO training loop
8. Reward plot
9. Post-training evaluation
10. Before vs After
11. Save model


In [ ]:
!pip -q install unsloth transformers trl accelerate bitsandbytes matplotlib

: 

In [ ]:
import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.optim import AdamW

from unsloth import FastLanguageModel

# Expected uploaded folders in Colab runtime:
# /content/env, /content/graders
sys.path.append("/content")

from env.environment import ExecAssistEnv
from env.models import ActionType, EmailCategory, ExecAssistAction

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE != "cuda":
    raise RuntimeError("This notebook expects a GPU runtime (Colab T4).")


In [ ]:
# Unsloth 4-bit model for Colab T4.
# For reliability we use a 3B checkpoint.
model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=1e-5)

print("Loaded with Unsloth:", model_name)
print("Trainable parameter tensors:", len(trainable_params))


In [ ]:
MAX_STEPS = {"easy": 10, "medium": 15, "hard": 12}
TASKS = ["easy", "medium", "hard"]

SYSTEM_PROMPT = """
You are an enterprise executive-assistant policy trained with reinforcement learning.
Return exactly one JSON object for the next action.

Valid action_type values:
- classify_email
- reply_email
- schedule_meeting
- ignore_email
- assign_task
- query_status
- update_project

Rules:
- Always include a valid email_id from pending_emails.
- classify_email needs category.
- reply_email needs reply_text.
- schedule_meeting needs meeting_start_time and meeting_end_time.
- assign_task needs team.
- update_project needs project_id and project_status.
Return JSON only.
""".strip()


def format_observation_as_text(obs: Dict[str, Any]) -> str:
    pending = obs.get("pending_emails", [])
    calendar = obs.get("calendar_events", [])
    world_state = obs.get("world_state", {})
    compact_obs = {
        "task_name": obs.get("task_name"),
        "current_step": obs.get("current_step"),
        "last_action_result": obs.get("last_action_result", ""),
        "pending_emails": [
            {
                "email_id": e.get("email_id"),
                "sender": e.get("sender"),
                "subject": e.get("subject"),
                "body": str(e.get("body", ""))[:220],
            }
            for e in pending
        ],
        "calendar_events": [
            {"title": c.get("title"), "start_time": c.get("start_time"), "end_time": c.get("end_time")}
            for c in calendar
        ],
        "world_state": {
            "projects": world_state.get("projects", []),
            "team_load": world_state.get("team_load", {}),
        },
    }
    return json.dumps(compact_obs, ensure_ascii=True)


def build_prompt(obs: Dict[str, Any]) -> str:
    return f"{SYSTEM_PROMPT}\n\nObservation:\n{format_observation_as_text(obs)}\n\nAction JSON:"


def _extract_json(text: str) -> Dict[str, Any]:
    cleaned = text.replace("```json", "").replace("```", "").strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start < 0 or end < 0 or end <= start:
        raise ValueError("No JSON object found")
    return json.loads(cleaned[start : end + 1])


def parse_action(action_text: str, obs: Dict[str, Any]) -> Dict[str, Any]:
    action = _extract_json(action_text)
    pending = obs.get("pending_emails", [])
    valid_ids = {e.get("email_id") for e in pending}

    valid_types = {
        ActionType.CLASSIFY_EMAIL.value,
        ActionType.REPLY_EMAIL.value,
        ActionType.SCHEDULE_MEETING.value,
        ActionType.IGNORE_EMAIL.value,
        ActionType.ASSIGN_TASK.value,
        ActionType.QUERY_STATUS.value,
        ActionType.UPDATE_PROJECT.value,
    }

    action_type = action.get("action_type")
    email_id = action.get("email_id")
    if action_type not in valid_types:
        raise ValueError("Invalid action_type")
    if email_id not in valid_ids:
        raise ValueError("Invalid email_id")

    if action_type == ActionType.CLASSIFY_EMAIL.value:
        allowed = {c.value for c in EmailCategory}
        if action.get("category") not in allowed:
            raise ValueError("classify_email missing/invalid category")

    if action_type == ActionType.REPLY_EMAIL.value:
        action.setdefault("reply_text", "Thanks for the message. I will follow up shortly.")

    if action_type == ActionType.SCHEDULE_MEETING.value:
        action.setdefault("meeting_title", "Meeting")
        action.setdefault("meeting_start_time", "2024-07-01 11:00")
        action.setdefault("meeting_end_time", "2024-07-01 11:30")
        if not action.get("participants"):
            sender = next((e.get("sender") for e in pending if e.get("email_id") == email_id), "unknown@example.com")
            action["participants"] = [sender]

    if action_type == ActionType.ASSIGN_TASK.value:
        action.setdefault("team", "engineering")

    if action_type == ActionType.UPDATE_PROJECT.value:
        action.setdefault("project_id", "P1")
        action.setdefault("project_status", "on_track")

    return action


def tokenize_prompt_action(prompt: str, action_text: str):
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)
    action_ids = tokenizer(action_text, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)
    full_ids = torch.cat([prompt_ids, action_ids], dim=1)
    return prompt_ids, action_ids, full_ids


def action_logprob(prompt: str, action_text: str) -> torch.Tensor:
    prompt_ids, action_ids, full_ids = tokenize_prompt_action(prompt, action_text)
    outputs = model(full_ids)
    logits = outputs.logits[:, :-1, :]
    target = full_ids[:, 1:]
    logp = F.log_softmax(logits, dim=-1)
    token_logp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1)

    p_len = prompt_ids.shape[1]
    a_len = action_ids.shape[1]
    start = max(p_len - 1, 0)
    end = start + a_len
    selected = token_logp[:, start:end]
    return selected.sum(dim=1)


def normalize(xs: List[float]) -> List[float]:
    if len(xs) == 0:
        return []
    t = torch.tensor(xs, dtype=torch.float32)
    if float(t.std().item()) < 1e-6:
        return [0.0 for _ in xs]
    t = (t - t.mean()) / (t.std() + 1e-6)
    return t.tolist()


@dataclass
class RolloutStep:
    prompt: str
    action_text: str
    reward: float
    old_logprob: float
    task: str
    seed: int


In [ ]:
# PPO-style clipped policy update config
PPO_CLIP_EPS = 0.2
PPO_EPOCHS_PER_BATCH = 2
BATCH_SIZE_STEPS = 8
TEMPERATURE = 0.8
MAX_NEW_TOKENS = 180


@torch.no_grad()
def generate_action_text(prompt: str, temperature: float = TEMPERATURE, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1536).to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # keep only generated suffix when possible
    if "Action JSON:" in text:
        text = text.split("Action JSON:", 1)[-1].strip()
    return text


def generate_and_parse_action(obs: Dict[str, Any], retry_once: bool = True) -> Tuple[Optional[Dict[str, Any]], Optional[str], Optional[str]]:
    prompt = build_prompt(obs)
    try:
        action_text = generate_action_text(prompt)
        action = parse_action(action_text, obs)
        return action, action_text, prompt
    except Exception:
        if retry_once:
            try:
                action_text = generate_action_text(prompt)
                action = parse_action(action_text, obs)
                return action, action_text, prompt
            except Exception:
                return None, None, prompt
        return None, None, prompt


def ppo_style_update(batch: List[RolloutStep]) -> float:
    if not batch:
        return 0.0

    FastLanguageModel.for_training(model)
    rewards = [x.reward for x in batch]
    advantages = normalize(rewards)

    total_loss = 0.0
    steps = 0

    for _ in range(PPO_EPOCHS_PER_BATCH):
        random.shuffle(batch)
        for item, adv in zip(batch, advantages):
            if item.action_text == "<parse_failed>":
                continue

            old_lp = torch.tensor(item.old_logprob, dtype=torch.float32, device=DEVICE)
            new_lp = action_logprob(item.prompt, item.action_text).squeeze(0)

            ratio = torch.exp(new_lp - old_lp)
            adv_t = torch.tensor(adv, dtype=torch.float32, device=DEVICE)

            unclipped = ratio * adv_t
            clipped = torch.clamp(ratio, 1.0 - PPO_CLIP_EPS, 1.0 + PPO_CLIP_EPS) * adv_t
            loss = -torch.min(unclipped, clipped)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()

            total_loss += float(loss.detach().cpu().item())
            steps += 1

    return total_loss / max(1, steps)


In [ ]:
def run_episode(task: str, seed: int, collect_for_train: bool) -> Tuple[float, List[RolloutStep]]:
    env = ExecAssistEnv(task_name=task, seed=seed)
    obs = env.reset().model_dump()
    done = False
    total_reward = 0.0
    rollout_steps: List[RolloutStep] = []

    safety_counter = 0
    while not done and safety_counter < (MAX_STEPS[task] * 3):
        safety_counter += 1

        action, action_text, prompt = generate_and_parse_action(obs, retry_once=True)
        if action is None or action_text is None or prompt is None:
            # safe step skip on parser failure
            rollout_steps.append(
                RolloutStep(
                    prompt=build_prompt(obs),
                    action_text="<parse_failed>",
                    reward=0.0,
                    old_logprob=0.0,
                    task=task,
                    seed=seed,
                )
            )
            break

        step_result = env.step(ExecAssistAction(**action)).model_dump()
        reward = float(step_result.get("reward", 0.0))
        done = bool(step_result.get("done", False))

        old_lp = float(action_logprob(prompt, action_text).detach().cpu().item()) if collect_for_train else 0.0
        rollout_steps.append(
            RolloutStep(
                prompt=prompt,
                action_text=action_text,
                reward=reward,
                old_logprob=old_lp,
                task=task,
                seed=seed,
            )
        )

        total_reward += reward
        obs = step_result["observation"]

    return total_reward, rollout_steps


def evaluate_policy(num_episodes: int = 4) -> Tuple[float, List[float]]:
    rewards: List[float] = []
    for i in range(num_episodes):
        task = random.choice(TASKS)
        seed = int(time.time()) + i + random.randint(1, 9999)
        ep_reward, _ = run_episode(task=task, seed=seed, collect_for_train=False)
        rewards.append(ep_reward)
        print(f"[EVAL] Episode {i+1}/{num_episodes} | task={task} | reward={ep_reward:.4f}")
    return sum(rewards) / max(1, len(rewards)), rewards


BASELINE_EPISODES = 4
TRAIN_EPISODES = 20
POST_EVAL_EPISODES = 4

print("=== Baseline Evaluation (No updates) ===")
before_avg_reward, baseline_rewards = evaluate_policy(num_episodes=BASELINE_EPISODES)
print(f"Baseline Avg Reward: {before_avg_reward:.4f}")

print("\n=== Unsloth RL Training (PPO-style clipped updates) ===")
reward_history: List[float] = []
loss_history: List[float] = []
trajectories: List[Dict[str, Any]] = []
train_buffer: List[RolloutStep] = []

for ep in range(1, TRAIN_EPISODES + 1):
    task = random.choice(TASKS)
    seed = int(time.time()) + ep + random.randint(1, 9999)
    ep_reward, rollout = run_episode(task=task, seed=seed, collect_for_train=True)

    reward_history.append(ep_reward)
    train_buffer.extend(rollout)

    trajectories.append(
        {
            "episode": ep,
            "task": task,
            "seed": seed,
            "reward": ep_reward,
            "steps": [
                {"reward": s.reward, "action": s.action_text[:220]}
                for s in rollout
            ],
        }
    )

    avg_loss = 0.0
    if len(train_buffer) >= BATCH_SIZE_STEPS:
        batch = train_buffer[:BATCH_SIZE_STEPS]
        train_buffer = train_buffer[BATCH_SIZE_STEPS:]
        avg_loss = ppo_style_update(batch)

    loss_history.append(avg_loss)
    print(f"[TRAIN] Episode {ep}/{TRAIN_EPISODES} | task={task} | reward={ep_reward:.4f} | loss={avg_loss:.6f}")

print("\n=== Post-Training Evaluation (No updates) ===")
after_avg_reward, post_rewards = evaluate_policy(num_episodes=POST_EVAL_EPISODES)
print(f"Final Avg Reward: {after_avg_reward:.4f}")

print("\n=== Before vs After ===")
print(f"Before: {before_avg_reward:.4f}")
print(f"After:  {after_avg_reward:.4f}")

if after_avg_reward <= before_avg_reward:
    print("No clear improvement yet; increase TRAIN_EPISODES or lower model size for more stable updates.")


In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(range(1, len(reward_history) + 1), reward_history, label="Train Reward", marker="o", linewidth=1.2)
plt.axhline(before_avg_reward, color="tab:orange", linestyle="--", label=f"Baseline Avg={before_avg_reward:.3f}")
plt.axhline(after_avg_reward, color="tab:green", linestyle="--", label=f"Post Avg={after_avg_reward:.3f}")
plt.title("OfficeAgentEnv Unsloth RL Reward Trend")
plt.xlabel("Episode")
plt.ylabel("Cumulative Reward")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(11, 3))
plt.plot(range(1, len(loss_history) + 1), loss_history, color="tab:red", marker=".")
plt.title("PPO-style Update Loss")
plt.xlabel("Episode")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.show()

save_dir = "/content/officeagent_rl_model/"
os.makedirs(save_dir, exist_ok=True)

# Save adapter + tokenizer (recommended for Unsloth/LoRA)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Saved model + tokenizer to: {save_dir}")
